# SPATIAL INTELLIGENCE - PART 1

In [1]:
# This cell is not needed if you have pip installed topologicpy
#import sys
#sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\Sushmitha\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.29) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [5]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## 5. Import the gallery floor plan

In [6]:
floor_plan = Topology.ByBREPPath(r"D:\3rd sem\GRAPH MACHINE LEARNING\ASSIGNMENTS\ASSIGNMENT-2\notebook-2\Graph-machine-learning\2.Resources\Market.brep")
triangles = Cluster.Faces(floor_plan)
shell = Shell.ByFaces(triangles)
eb = Shell.ExternalBoundary(shell)
ib_list = Shell.InternalBoundaries(shell)
new_face = Face.ByWires(eb, ib_list)
market = Topology.RemoveCollinearEdges(new_face)
print(market)



## 6. Show the geometry

In [7]:
Topology.Show(market,
              camera=[0,0,7],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Create a grid overlay

In [8]:
faces = Topology.Faces(market)

print("Number of faces:", len(faces))

Topology.Faces - Warning: The input is a Face. Returning the same face embedded in a list.
caller name: <module>
Number of faces: 1


In [9]:
b_r = Wire.BoundingRectangle(market)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+6,6))
vRange = list(range(0,int(length)+6,6))

grid = Grid.EdgesByDistances(market, clip=True, uRange=uRange, vRange=vRange)

## 8. Show the geometry and the grid

In [10]:
Topology.Show(market, grid,
              camera=[0,0,7],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 9. Slice the floor plan with the grid to create a topologic shell

In [11]:
shell = Topology.Slice(market, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## 10. Show the resulting shell

In [12]:
Topology.Show(shell,
              camera=[0,0,7],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 11. Derive navigation and analysis graphs from the shell

In [13]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## 12. Derive and store the analysis graph vertices

In [14]:
g_verts = Graph.Vertices(analysis_graph)

## 13. Show the analysis graph

In [15]:
Topology.Show(analysis_graph, 
              camera=[0,0,7],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

## ORIGINAL GRAPH ANALYSIS

In [16]:
density_original = Graph.Density(
    analysis_graph
)

diameter_original = Graph.Diameter(
    analysis_graph
)

print("ORIGINAL GRAPH")
print("Density :", density_original)
print("Diameter:", diameter_original)

Topology.Show(
    analysis_graph,
    camera=[0,0,7],
    vertexSize=8,
    vertexColor="red",
    edgeColor="lightgrey",
    edgeWidth=2,
    backgroundColor="black",
    width=900,
    height=700,
    renderer=renderer
)

ORIGINAL GRAPH
Density : 0.006039676144430375
Diameter: 41


## 14. Spatial Intelligence through Graph Analysis

### a. Minimum Spanning Tree
MST is very time consuming to compute on a large grid graph. Here we will demonstrate it on a simple graph

In [17]:
cc1 = CellComplex.Prism()
cc2 = Topology.Translate(cc1, 1.1, 0, 0)
cc3 = Topology.Translate(cc2, 1.1, 0, 0)
g1 = Graph.ByTopology(cc1)
g2 = Graph.ByTopology(cc2)
g3 = Graph.ByTopology(cc3)
g2 = Graph.MinimumSpanningTree(g2)
g3 = Graph.Complete(g3)
Topology.Show(g1, g2, g3,
              vertexSize=12,
              vertexColor="red",
              edgeColor="lightgrey",
              edgeWidth=4,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)


In [18]:
dn1 = Graph.Density(g1)
dn2 = Graph.Density(g2)
dn3 = Graph.Density(g3)
print("Density 1:", dn1)
print("Density 2:", dn2)
print("Density 3:", dn3)

Density 1: 0.42857142857142855
Density 2: 0.25
Density 3: 1.0


In [19]:
dr1 = Graph.Diameter(g1)
dr2 = Graph.Diameter(g2)
dr3 = Graph.Diameter(g3)
print("Diameter 1:", dr1)
print("Diameter 2:", dr2)
print("Diameter 3:", dr3)

Diameter 1: 3
Diameter 2: 5
Diameter 3: 1


## MINIMUM SPANNING TREE ANALYSIS

In [ ]:
mst_graph = Graph.MinimumSpanningTree(
    analysis_graph
)

density_mst = Graph.Density(
    mst_graph
)

diameter_mst = Graph.Diameter(
    mst_graph
)
print("MINIMUM SPANNING TREE")
print("Density :", density_mst)
print("Diameter:", diameter_mst)


AttributeError: 'topologic_core.Graph' object has no attribute 'vertex_count'

### b. Shortest Path (Use navigation graph)

In [ ]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=market)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

Shortest Path Duration: 0.71 seconds
Straighten Wire Duration: 31.32 seconds
Original Shortest Path Length: 229.39
Straightened Shortened Path Length: 206.21


In [ ]:
Topology.Show(new_face, shortest_path, straight_path,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

#### CALCULATION OF SHORTEST PATH ANALYSIS

In [ ]:
# ==========================================
# SHORTEST PATH ANALYSIS
# ==========================================

import time

# start + end points
start_vertex = Vertex.ByCoordinates(
    xmin+2,
    ymax-2,
    0
)

end_vertex = Vertex.ByCoordinates(
    xmax-2,
    ymin+2,
    0
)

# compiled routing graph
crg = Graph.CompiledRoutingGraph(
    navigation_graph,
    precomputeTurns=False
)

# shortest path
start = time.time()

shortest_path = Graph.ShortestPath(
    crg,
    vertexA=start_vertex,
    vertexB=end_vertex
)

end = time.time()

print("Shortest Path Duration:",
      round(end-start,2),
      "seconds")

# straighten path
straight_path = Wire.Straighten(
    shortest_path,
    host=market
)

# lengths
original_length = Wire.Length(shortest_path)

straight_length = Wire.Length(straight_path)

# edges
sp_edges = Topology.Edges(shortest_path)

# turns
num_turns = max(
    len(Topology.Vertices(shortest_path))-2,
    0
)

print("\nShortest Path")
print("Original SP Length:",
      round(original_length,2))

print("Straightened SP Length:",
      round(straight_length,2))

print("Number of edges in path:",
      len(sp_edges))

print("Number of turns:",
      num_turns)

Shortest Path Duration: 0.62 seconds

Shortest Path
Original SP Length: 229.39
Straightened SP Length: 206.21
Number of edges in path: 80
Number of turns: 79


### c. Closeness Centrality/Integration
* Closeness centrality is a graph metric that quantifies how close a node is to all other nodes by taking the reciprocal of the sum of its shortest path distances to every other node in the network.
* In space syntax, closeness centrality corresponds to global integration, measuring how spatially accessible or topologically shallow a space is within a configuration, thereby indicating its potential for movement flow and encounter density.

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

#### CALCULATIONS OF CLOSENESS CENTRALITY ANALYSIS + VISUALIZATION

In [ ]:
centrality_list = Graph.BetweennessCentrality(
    analysis_graph,
    normalize=True,
    colorScale="thermal"
)

In [ ]:
# =========================================================
# CLOSENESS CENTRALITY ANALYSIS + VISUALIZATION
# =========================================================

from statistics import mean

from topologicpy.Graph import Graph
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Color import Color

# ---------------------------------------------------------
# REBUILD GRAPH (SAFE)
# ---------------------------------------------------------

analysis_graph = Graph.ByTopology(shell)

print("Graph Created")

# ---------------------------------------------------------
# CALCULATE CLOSENESS CENTRALITY
# ---------------------------------------------------------

# IMPORTANT:
# do NOT assign result back to analysis_graph

Graph.ClosenessCentrality(
    analysis_graph,
    key="closeness"
)

print("Centrality Calculated")

# ---------------------------------------------------------
# GET GRAPH VERTICES
# ---------------------------------------------------------

g_verts = Graph.Vertices(analysis_graph)

print("Number of vertices:", len(g_verts))

# ---------------------------------------------------------
# EXTRACT CENTRALITY VALUES
# ---------------------------------------------------------

cc_values = []

for v in g_verts:

    d = Topology.Dictionary(v)

    if not d:
        continue

    cc = Dictionary.ValueAtKey(
        d,
        "closeness"
    )

    if cc is not None:
        cc_values.append(cc)

# ---------------------------------------------------------
# CHECK VALUES
# ---------------------------------------------------------

if len(cc_values) == 0:

    print("No closeness values found")

else:

    # -----------------------------------------------------
    # STATISTICS
    # -----------------------------------------------------

    cc_min = min(cc_values)
    cc_max = max(cc_values)
    cc_mean = mean(cc_values)

    print("\nCloseness Centrality Statistics")
    print("Min :", round(cc_min,4))
    print("Max :", round(cc_max,4))
    print("Mean:", round(cc_mean,4))
    print("Spaces analysed:", len(cc_values))

    # -----------------------------------------------------
    # COLOR VERTICES
    # -----------------------------------------------------

    for v in g_verts:

        d = Topology.Dictionary(v)

        if not d:
            continue

        cc = Dictionary.ValueAtKey(
            d,
            "closeness"
        )

        if cc is None:
            continue

        color = Color.AnyToHex(
            Color.ByValueInRange(
                cc,
                minValue=cc_min,
                maxValue=cc_max,
                colorScale="thermal"
            )
        )

        d = Dictionary.SetValueAtKey(
            d,
            "cc_color",
            color
        )

        d = Dictionary.SetValueAtKey(
            d,
            "size",
            12
        )

        v = Topology.SetDictionary(v, d)

    print("Vertex Colors Assigned")

# ---------------------------------------------------------
# TRANSFER TO FACES
# ---------------------------------------------------------

faces = Topology.Faces(shell)

_ = transfer_dicts_by_key(
    faces,
    g_verts,
    "face_id"
)

print("Face Dictionaries Updated")

# ---------------------------------------------------------
# VISUALIZE
# ---------------------------------------------------------

Topology.Show(
    faces,

    faceColorKey="cc_color",
    faceOpacity=1,

    showEdges=False,
    showVertices=False,

    vertexSizeKey="size",

    camera=[0,0,7],

    backgroundColor="black",

    width=1000,
    height=700,

    renderer=renderer
)

### d. Betweenness Centrality/Choice
* Betweenness centrality measures how often a node lies on the shortest paths between other nodes.

In [ ]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

* Transfer the information from the graph back to the shell

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)